In [3]:
import pandas as pd

path = r'C:\Users\AVINASH\Downloads\DataScience Assessment'

captains = pd.read_csv(path + r'\captains.csv')
approvals = pd.read_csv(path + r'\approvals.csv')
nudges = pd.read_csv(path + r'\nudges.csv')

captains['signup_ts'] = pd.to_datetime(captains['signup_ts'])
nudges['sent_ts'] = pd.to_datetime(nudges['sent_ts'])

# May 2026 cohort
cohort = captains[
    (captains['signup_ts'] >= '2026-05-01') &
    (captains['signup_ts'] < '2026-06-01')
].copy()

# Identify CAMP_WA_002 correctly
wa002 = nudges[
    (nudges['campaign_id'] == 'CAMP_WA_002') &
    (nudges['captain_id'].isin(cohort['captain_id']))
]

# Mark targeted captains
targeted_ids = set(wa002['captain_id'])
cohort['targeted'] = cohort['captain_id'].isin(targeted_ids)

# Add approval status
cohort = cohort.merge(
    approvals[['captain_id', 'final_status']],
    on='captain_id',
    how='left'
)

cohort['approved'] = (cohort['final_status'] == 'approved').astype(int)

# Targeted vs non-targeted
summary = cohort.groupby('targeted').agg(
    captains=('captain_id', 'count'),
    approved=('approved', 'sum')
)

summary['A2O_%'] = (
    summary['approved'] / summary['captains'] * 100
).round(2)

print("CAMP_WA_002 — Targeted vs Non-targeted")
print(summary)

# Observed difference
targeted_rate = summary.loc[True, 'A2O_%']
control_rate = summary.loc[False, 'A2O_%']
difference = targeted_rate - control_rate

print(f"\nObserved A2O difference: {difference:.2f} percentage points")

# Campaign engagement
print("\nCampaign engagement:")
print(
    wa002.groupby('channel').agg(
        sent=('captain_id', 'count'),
        delivered=('delivered', 'sum'),
        clicked=('clicked', 'sum')
    )
)

# Timing
timing = wa002.groupby('captain_id')['sent_ts'].min().reset_index()
timing = timing.merge(
    cohort[['captain_id', 'signup_ts']],
    on='captain_id'
)
timing['days_after_signup'] = (
    timing['sent_ts'] - timing['signup_ts']
).dt.total_seconds() / 86400

print("\nCampaign timing:")
print(timing['days_after_signup'].describe())

# Observational incremental approvals
targeted_count = summary.loc[True, 'captains']
targeted_approved = summary.loc[True, 'approved']

expected_without_campaign = targeted_count * control_rate / 100
incremental = targeted_approved - expected_without_campaign

print(f"\nTargeted captains: {targeted_count}")
print(f"Actual targeted approvals: {targeted_approved}")
print(f"Estimated approvals without campaign: {expected_without_campaign:.1f}")
print(f"Estimated incremental approvals: {incremental:.1f}")

print("\nNOTE: This is an observational association, not a causal effect.")

                total_captains  approved  dropped  approval_rate
targeted_wa002                                                  
False                    16327      1733    13489      10.614320
True                      8673      2473     5573      28.513778

Click performance for CAMP_WA_002:
         total  approved  approval_rate
clicked                                
0         5104      1465      28.702978
1         3569      1008      28.243205
